# S2 · Theory checks: known posterior, then real patients

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/sct-range/blob/main/notebooks/s2_synthetic_benchmark.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hashirama21/sct-range/blob/main/notebooks/s2_synthetic_benchmark.ipynb)

Week 2 tests the theory of `docs/theory.md` where the truth is known exactly: stationary random fields with an exact
covariance, and a voxel-wise tissue-mixture posterior whose marginals are closed-form and whose dependence is set by a
Gaussian copula. Section 7 then checks, on real SynthRAD2025 patients, the statements that are identities and
therefore hold for any error field, and measures the quantities they involve. Every statement is checked against
an explicit criterion; section 8 collects the verdicts.

| Section | Statement |
|---|---|
| 1 | Propositions 1 and 1′: sharp rearrangement bounds; Proposition 2 and Corollary 2.1: at a fixed MAE, range error is governed by error coherence |
| 2 | Propositions 3 and 3.1: range error is low-pass along the beam; all angles weight errors by 1/\|k\| |
| 3 | Propositions 4 and 5: the Bayes act for range is the SPR mean; L1 creates a signed range bias |
| 4 | Theorem 6: only the mean is consistent under projection |
| 5 | Proposition 7: tails of range error need the joint law; the FZ score elicits VaR and ES |
| 6 | Proposition 8: projected CRPS separates dependence structures that voxel-wise CRPS cannot |
| 7 | Propositions 1, 1′, 2 and 3 on the SPR errors of real synthetic CTs (SynthRAD2025 development cases) |

Units: 1 voxel = 1 mm along the beam; SPR is dimensionless; range errors are in mm of water.

The notebook runs the code of this repository (`src/`), whether or not the package is installed.

**Synthetic where the truth is needed, real everywhere else.** Sections 1–6 compare measured quantities with their
exact theoretical values, which requires the posterior law of the CT given the source image; no patient dataset
provides it. Section 7 runs on the real SynthRAD2025 training data (development cases of the frozen split of S1,
fetched from Hugging Face if missing): the bulk-density baseline and every out-of-fold prediction already produced by S3 or S4.
The statements that need a posterior are tested on patients, without a known truth, in S3 (Proposition 5) and S4.

### Setup · install `sctrange` from the repository

Run this cell first. Inside a clone of [hashirama21/sct-range](https://github.com/hashirama21/sct-range) it installs that
clone in editable mode (`pip install -e ".[dose]"`), so the notebook always runs the repository's code, local edits
included. Opened on its own, from the Colab or Kaggle badge or on another machine, it first downloads the repository
from GitHub (no git needed) to `/content/sct-range` on Colab, `/kaggle/tmp/sct-range` on Kaggle (so that
`/kaggle/working` only receives outputs) or `./sct-range` elsewhere, then installs it. On Kaggle, enable
*Settings → Internet* first. `SCTRANGE_REF` selects a branch, tag or commit (default `main`), `SCTRANGE_HOME` the
destination; pip only runs when `sctrange` is missing or installed from elsewhere (`SCTRANGE_REINSTALL=1` forces it).

In [ ]:
import importlib
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/hashirama21/sct-range"
REPO_REF = os.environ.get("SCTRANGE_REF", "main")
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()


def find_repo(start: Path) -> Path | None:
    return next((p for p in (start, *start.parents) if (p / "pyproject.toml").exists() and (p / "src" / "sctrange").is_dir()),
                None)


def download_repo(destination: Path) -> Path:
    """Fetch ``REPO_REF`` of the GitHub repository as a zip archive and unpack it to ``destination``."""
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f"{REPO_URL}/archive/{REPO_REF}.zip", timeout=120) as response:
                archive = zipfile.ZipFile(io.BytesIO(response.read()))
            break
        except OSError as error:
            if attempt == 3:
                raise
            print(f"download failed ({error}); retrying")
            time.sleep(5 * (attempt + 1))
    top = archive.namelist()[0].split("/")[0]
    staging = destination.with_name(destination.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    archive.extractall(staging)
    destination.parent.mkdir(parents=True, exist_ok=True)
    (staging / top).rename(destination)
    shutil.rmtree(staging, ignore_errors=True)
    return destination


REPO_ROOT = find_repo(Path.cwd())
if REPO_ROOT is None:
    default_home = (Path("/content/sct-range") if ON_COLAB else Path("/kaggle/tmp/sct-range") if ON_KAGGLE
                    else Path.cwd() / "sct-range")
    REPO_ROOT = Path(os.environ.get("SCTRANGE_HOME", default_home)).resolve()
    if find_repo(REPO_ROOT) is None:
        print(f"downloading {REPO_URL} @ {REPO_REF} to {REPO_ROOT}")
        download_repo(REPO_ROOT)
    os.chdir(REPO_ROOT / "notebooks")
SOURCE_DIR = (REPO_ROOT / "src").resolve()


def installed_from_repo() -> bool:
    spec = importlib.util.find_spec("sctrange")
    return spec is not None and spec.origin is not None and Path(spec.origin).resolve().is_relative_to(SOURCE_DIR)


if os.environ.get("SCTRANGE_REINSTALL") == "1" or not installed_from_repo():
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{REPO_ROOT}[dose]"]
    for attempt in range(1, 4):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0:
            break
        print(f"pip install failed (attempt {attempt}/3):\n{result.stderr[-3000:]}")
        time.sleep(10 * attempt)
    else:
        raise RuntimeError("pip could not install sctrange; see the messages above (network? Kaggle: enable Internet)")
# an editable install is only picked up by new processes; make it visible to this kernel too
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
for module in [name for name in sys.modules if name == "sctrange" or name.startswith("sctrange.")]:
    del sys.modules[module]
importlib.invalidate_caches()

import sctrange

assert Path(sctrange.__file__).resolve().is_relative_to(SOURCE_DIR), f"sctrange imported from {sctrange.__file__}"
print(f"sctrange {sctrange.__version__} installed from {REPO_ROOT.name}/src (editable)")

### Parameters

Set them here and run all cells (*Runtime → Run all* on Colab, where this cell is a form). Environment variables
of the same meaning, given in brackets, override them for unattended runs (`jupyter nbconvert --execute`).

Sections 1–6 need no dataset and run in about half a minute on a CPU. Section 7 needs the real data of `TASK`
(fetched from Hugging Face if missing) and the frozen split of S1: run S1 first.
| Platform | Dataset, cache | Outputs | GPU |
|---|---|---|---|
| local | `data/synthrad2025`, `outputs/` | `outputs/` | every visible GPU |
| Colab | `/content/data`, `/content/cache` | Google Drive `MyDrive/sct-range/outputs` (`USE_DRIVE`) | *Runtime → Change runtime type → T4 GPU* |
| Kaggle | `/kaggle/tmp/data`, `/kaggle/tmp/cache` | `/kaggle/working/outputs` | *Settings → Accelerator → GPU T4 × 2*, *Internet on* |

With two GPUs (Kaggle T4 × 2) the remaining folds train in parallel, one per GPU. T4 GPUs train in float16 with loss
scaling (they have no native bfloat16). Predictions are stored as int16 HU to fit the 20 GB of `/kaggle/working`.

In [ ]:
# @title Parameters { display-mode: "form" }
TASK = "1"  # @param ["1", "2"]  section 7: 1 = MRI, 2 = CBCT [SCT_TASK]
DOWNLOAD = True  # @param {type:"boolean"}  fetch missing data from Hugging Face [SCT_DOWNLOAD]
N_CASES = 20  # @param {type:"integer"}  development cases in section 7, 0 = all [SCT_MAX_CASES]
USE_DRIVE = True  # @param {type:"boolean"}  Colab only

In [ ]:
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sctrange.benchmark import (
    angle_energy_experiment,
    autocorrelation_experiment,
    estimator_experiment,
    fz_check,
    paired_difference,
    projected_crps_experiment,
    projection_consistency_table,
    rearrangement_experiment,
    summarize_estimators,
    tail_experiment,
    transfer_experiment,
)
from sctrange.data import bulk_density_sct, discover_cases, load_case, read_volume
from sctrange.download import ensure_dataset
from sctrange.empirical import case_identities, summarize_identities
from sctrange.fields import gaussian_random_field
from sctrange.hlut import HLUT
from sctrange.plotting import (
    apply_style,
    plot_angle_energy,
    plot_autocorrelation_law,
    plot_estimator_bias,
    plot_spectral_weights,
    plot_tail_ecdf,
)
from sctrange.spectral import segment_energy_spectral, sliding_segment_sums
from sctrange.split import load_split

In [ ]:
import os

from sctrange.runtime import frozen_split_path, resolve_paths

PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASK = os.environ.get("SCT_TASK", TASK)
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
N_CASES = int(os.environ.get("SCT_MAX_CASES", N_CASES)) or None
OUTPUT_DIR = PATHS.output_root / "s2"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20261013
CORRELATION_LENGTHS = (0, 1, 2, 4, 8, 16, 32, 64)
TAIL_CORRELATION_LENGTHS = (0, 4, 16)
SEGMENT_LENGTHS = (10, 60)
ANGLE_FREQUENCIES = np.geomspace(0.02, 0.15, 8)
LABELS = {"range, L = 10 mm": (0.447, 10), "range, L = 60 mm": (0.155, 10), "gradient loss": (0.2, -14), "voxel L2": (0.12, 10)}

hlut = HLUT.pyradplan_default()
checks: list[tuple[str, str, str, bool]] = []


def rng(section: int) -> np.random.Generator:
    return np.random.default_rng([SEED, section])


def record(section: str, claim: str, evidence: str, passed: bool) -> None:
    checks.append((section, claim, evidence, bool(passed)))
    print(f"[{'PASS' if passed else 'FAIL'}] {claim}: {evidence}")


apply_style()
pd.set_option("display.width", 160)

## 1 · Coherence, not magnitude, sets the range error

**Rearrangements.** Seven continuous errors of both signs are placed on seven voxels, three of which lie on the ray.
All 5040 arrangements are enumerated. Propositions 1 and 1′ predict that the extremes equal the sorted (Hardy–Littlewood)
bounds and that only arrangements ordering the errors like the weights reach the maximum (Bellini et al., 2022,
Lemma A.2).

In [ ]:
rearrangement = rearrangement_experiment(rng(12))
bound_gap = max((rearrangement["exhaustive_max"] - rearrangement["bound_max"]).abs().max(),
                (rearrangement["exhaustive_min"] - rearrangement["bound_min"]).abs().max())
record("1", "Propositions 1, 1′: exhaustive extremes equal the sorted bounds", f"max gap {bound_gap:.1e}", bound_gap < 1e-12)
record("1", "Only comonotone arrangements attain the maximum",
       f"{rearrangement['all_maximisers_comonotone'].sum()}/{len(rearrangement)} trials, "
       f"min gap to next value {rearrangement['gap_to_second_value'].min():.3f}",
       rearrangement["all_maximisers_comonotone"].all() and (rearrangement["gap_to_second_value"] > 0).all())
display(rearrangement.head().round(4))

**Autocorrelation law.** SPR errors are stationary Gaussian fields with standard deviation 0.03 (≈ 30 HU in soft tissue) and an exponential
correlation of length λ along and across the beam. The marginal law, hence the MAE, is identical for every λ.
Proposition 2 predicts the RMS error over a 100 mm segment from the exact circular covariance. The control shuffles
the same error values across all voxels and fields: identical MAE, white layout.

In [ ]:
coherence = autocorrelation_experiment(CORRELATION_LENGTHS, rng(1))
biased = autocorrelation_experiment((0, 8), rng(11), mean=0.002)

relative_error = (coherence["rms_observed_mm"] / coherence["rms_predicted_mm"] - 1).abs()
record("1", "Proposition 2 predicts RMS ΔWET", f"max relative error {relative_error.max():.1%}", relative_error.max() < 0.08)
record("1", "Bias term L²μ² adds as predicted",
       f"max relative error {(biased['rms_observed_mm'] / biased['rms_predicted_mm'] - 1).abs().max():.1%}",
       (biased["rms_observed_mm"] / biased["rms_predicted_mm"] - 1).abs().max() < 0.08)
mae_spread = coherence["mae_spr"].std() / coherence["mae_spr"].mean()
record("1", "MAE unchanged across correlation lengths", f"coefficient of variation {mae_spread:.1%}", mae_spread < 0.05)
gain = coherence["rms_observed_mm"].max() / coherence["rms_observed_mm"].min()
record("1", "Corollary 2.1: same MAE, several-fold range error", f"RMS ratio {gain:.1f}×", gain > 5)
shuffled = (coherence["rms_permuted_mm"] / coherence["rms_predicted_mm"].iloc[0] - 1).abs().max()
record("1", "Shuffled errors behave as white noise", f"max deviation {shuffled:.1%}", shuffled < 0.05)

display(coherence.round(4))
plot_autocorrelation_law(coherence).savefig(FIGURE_DIR / "fig1_coherence.png", bbox_inches="tight")
plt.show()

## 2 · Spectral weighting

**Segments.** Plane-wave errors along the beam are summed over circular segments of length L. Proposition 3 states an
exact identity: the measured transfer must equal the Dirichlet kernel to machine precision, and errors that vary only
across the beam must pass unattenuated. Voxel-wise L2 and a forward-difference gradient loss are shown for comparison.

In [ ]:
fields = gaussian_random_field((128, 48), 8.0, 4, rng(2))
parseval = max(
    abs(np.sum(sliding_segment_sums(field, length, axis=0) ** 2) - segment_energy_spectral(field, length, axis=0))
    / np.sum(sliding_segment_sums(field, length, axis=0) ** 2)
    for field in fields for length in (*SEGMENT_LENGTHS, 30)
)
record("2", "Proposition 3: Parseval identity on random fields", f"max relative error {parseval:.1e}", parseval < 1e-10)

transfer = transfer_experiment(SEGMENT_LENGTHS)
transfer_gap = float(np.max(np.abs(transfer["measured"] - transfer["theory"])))
record("2", "Measured transfer equals theory", f"max absolute gap {transfer_gap:.1e}", transfer_gap < 1e-10)
across = transfer[transfer["functional"] == "range, across beam"]["measured"]
record("2", "Errors varying across the beam pass unattenuated", f"transfer in [{across.min():.6f}, {across.max():.6f}]",
       np.allclose(across, 1.0))

landmarks = transfer[transfer["frequency"].isin([2 / 512, 8 / 512, 32 / 512, 128 / 512, 256 / 512])]
display(landmarks.pivot(index="frequency", columns="functional", values="theory").round(5))
plot_spectral_weights(transfer, LABELS).savefig(FIGURE_DIR / "fig2a_spectral_weights.png", bbox_inches="tight")
plt.show()

**All angles.** Windowed plane waves are projected with the differentiable projector over 180 angles. Proposition 3.1
predicts that the angle-summed range energy, relative to the voxel energy, decays as 1/|k|: a log-log slope of −1.

In [ ]:
angles, slope = angle_energy_experiment(ANGLE_FREQUENCIES, rng(3))
record("2", "Proposition 3.1: angle-summed energy ∝ 1/|k|", f"log-log slope {slope:.3f}", abs(slope + 1) < 0.15)
display(angles.round(3))
plot_angle_energy(angles, slope).savefig(FIGURE_DIR / "fig2b_angle_energy.png", bbox_inches="tight")
plt.show()

## 3 · Bayes acts under range losses

Each voxel is soft tissue, or with probability pᵢ an alternative tissue (gas, or bone in the second scenario); pᵢ is a
smooth map of blobs peaking at 0.45, and a copula of correlation length 8 mm couples the classes. Three point estimates are
computed from the exact marginals: the HU mean (Bayes act of L2 in HU), the HU median (Bayes act of L1, in HU or SPR) and
the SPR mean g⁻¹(E[g(Y)]), the Bayes act of the squared range loss (Proposition 4). Segments run from depth 0 to 40, 80 and
120 mm, one per column.

For every ray the Monte Carlo risk over 3000 posterior draws is compared with the analytic decomposition
risk = bias² + variance, where the bias is computed from the exact marginals alone.

In [ ]:
estimator_detail = pd.concat([estimator_experiment(hlut, scenario, rng(4 + i)) for i, scenario in enumerate(("gas", "bone"))])
estimator_summary = summarize_estimators(estimator_detail)

bayes = estimator_summary.xs("SPR mean (Bayes act)", level="estimator")
record("3", "Bayes act has zero range bias", f"max |mean bias| {bayes['mean_bias_mm'].abs().max():.1e} mm",
       bayes["mean_bias_mm"].abs().max() < 1e-9)
lowest = estimator_summary.groupby(["scenario", "depth_mm"])["rms_mc_mm"].idxmin().map(lambda index: index[1])
record("3", "Bayes act has the lowest Monte Carlo range risk", f"{(lowest == 'SPR mean (Bayes act)').sum()}/{len(lowest)} settings",
       (lowest == "SPR mean (Bayes act)").all())
decomposition = (estimator_summary["rms_mc_mm"] / estimator_summary["rms_predicted_mm"] - 1).abs().max()
record("3", "Risk = analytic bias² + variance", f"max relative gap {decomposition:.1%}", decomposition < 0.02)
median_bias = estimator_summary.xs("HU median (L1)", level="estimator")["mean_bias_mm"]
record("3", "Proposition 5: L1 over-estimates SPR next to uncertain gas",
       f"mean bias {median_bias.loc['gas'].min():.1f} to {median_bias.loc['gas'].max():.1f} mm", (median_bias.loc["gas"] > 0).all())
record("3", "Proposition 5: L1 under-estimates SPR next to uncertain bone",
       f"mean bias {median_bias.loc['bone'].min():.1f} to {median_bias.loc['bone'].max():.1f} mm", (median_bias.loc["bone"] < 0).all())

display(estimator_summary.round(3))
plot_estimator_bias(estimator_detail[estimator_detail["depth_mm"] == 120]).savefig(
    FIGURE_DIR / "fig3a_estimator_bias.png", bbox_inches="tight")
plt.show()

## 4 · Only the mean is consistent under projection

Six copies of the uniform law on a small HU multiset are coupled either cyclically (the joint mix of Theorem 6, whose sum
is constant) or independently (the law of the sum is enumerated exactly over 6⁶ outcomes). A functional is
projection-consistent when its value on the sum equals the sum of its voxel values under every coupling.

In [ ]:
consistency = projection_consistency_table([-850.0, 40.0, 40.0, 40.0, 40.0, 700.0]).set_index("functional")
couplings = ["value on sum, cyclic coupling", "value on sum, independent coupling"]
matches = consistency[couplings].sub(consistency["sum of voxel values"], axis=0).abs().max(axis=1) < 1e-9
record("4", "Mean is additive under both couplings", f"max gap {consistency.loc['mean', couplings].sub(consistency.loc['mean', 'sum of voxel values']).abs().max():.1e}",
       matches["mean"])
record("4", "Median, quantile and expectiles are not", f"{(~matches.drop('mean')).sum()}/{len(matches) - 1} inconsistent",
       (~matches.drop("mean")).all())
display(consistency.round(2))

## 5 · Tails of range error need the joint law

The gas posterior is sampled with three copula correlation lengths. Voxel marginals, and therefore the Bayes act, are
identical in all three; only the dependence between voxels changes. The full-depth range error of the same point estimate
is then compared. The FZ0 score, minimised over a grid that does not contain the empirical values, must recover the
empirical VaR and ES of |ΔWET| within one grid step.

In [ ]:
tails, tail_errors = tail_experiment(hlut, TAIL_CORRELATION_LENGTHS, rng(6))
fraction_gap = tails["alternative_fraction"].max() - tails["alternative_fraction"].min()
record("5", "Identical marginals across copulas", f"gas fraction range {fraction_gap:.3f}", fraction_gap < 0.01)
var_ratio = tails["VaR90_mm"].iloc[-1] / tails["VaR90_mm"].iloc[0]
record("5", "Proposition 7: same point estimate, different tail", f"VaR90 ratio {var_ratio:.1f}×", var_ratio > 2)

fz = fz_check(tail_errors[TAIL_CORRELATION_LENGTHS[-1]], rng=rng(7))
fz_ok = (abs(fz["FZ argmin VaR (mm)"] - fz["empirical VaR (mm)"]) <= 1.01 * fz["grid step VaR (mm)"]
         and abs(fz["FZ argmin ES (mm)"] - fz["empirical ES (mm)"]) <= 1.01 * fz["grid step ES (mm)"])
record("5", "FZ0 score elicits (VaR, ES) of |ΔWET|", "within one grid step", fz_ok)

display(tails.round(3))
display(pd.Series(fz, name="value").to_frame().round(3))
plot_tail_ecdf(tail_errors).savefig(FIGURE_DIR / "fig3b_tails.png", bbox_inches="tight")
plt.show()

## 6 · Evaluating generative predictions

Observations are drawn from the gas posterior with copula length 16 mm. Two ensemble forecasts with the exact voxel
marginals are scored: the correct model, and a model with independent voxels. Voxel-wise CRPS averages one-dimensional
scores per voxel; projected CRPS scores the law of each full-depth range integral. Differences are paired over
observations, with percentile bootstrap intervals.

In [ ]:
crps = projected_crps_experiment(hlut, rng(8))
voxel = paired_difference(crps, "voxel CRPS, correct", "voxel CRPS, wrong dependence", rng(9))
projected = paired_difference(crps, "projected CRPS, correct", "projected CRPS, wrong dependence", rng(10))
record("6", "Voxel CRPS cannot tell the two forecasts apart",
       f"95% CI [{voxel['ci_low']:.4f}, {voxel['ci_high']:.4f}] contains 0", voxel["ci_low"] < 0 < voxel["ci_high"])
record("6", "Projected CRPS penalises the wrong dependence",
       f"95% CI [{projected['ci_low']:.2f}, {projected['ci_high']:.2f}] mm, {projected['relative to score']:.0%} worse",
       projected["ci_low"] > 0)
display(pd.DataFrame({"voxel CRPS (SPR)": voxel, "projected CRPS (mm)": projected}).T.round(4))

## 7 · The identities on real patients

Propositions 1, 1′, 2 and 3 are identities: they hold for every error field, so they can be checked exactly on real
data. For each development case of `TASK` (frozen split of S1; test cases are never loaded), the SPR error of every
available synthetic CT — the bulk-density baseline of the challenge and any out-of-fold prediction of S3 or S4 — is
taken on 16 slices around the largest body section, with beams along the two in-plane axes (0° and 90°) and circular
segments of 10 and 60 mm.

- **Proposition 3**: the energy of segment sums equals the Dirichlet-weighted spectrum of the real error;
- **Proposition 2**: with the mean μ, variance σ² and autocorrelation measured on the same error, L²μ² + Lσ²l is the
  mean squared segment error; the effective length *l* (1 sample for white errors) is the coherence of real errors;
- **Propositions 1, 1′**: the range error of every ray of the largest slice lies between the sums of the *n* smallest
  and *n* largest voxel errors of that slice; its *position* between them (0 = lower, 1 = upper bound) shows how close
  the real layout comes to the worst case.

`coherence gain` = RMS segment error / RMS of white errors with the same μ and σ: the factor by which the spatial
layout of real errors, not their magnitude, inflates range error (Corollary 2.1).

In [ ]:
DATA_ROOT = ensure_dataset(PATHS.data_root, (TASK,), download=DOWNLOAD)
SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"{SPLIT_PATH} is missing: run the S1 notebook once on the real data to freeze it")
split = load_split(SPLIT_PATH)
development = [case for case in discover_cases(DATA_ROOT)
               if case.task == TASK and case.case_id in set(split["development"])][:N_CASES]
if not development:
    raise RuntimeError(f"no development case of task {TASK} under {DATA_ROOT}")
PREDICTIONS = {**{f"S3 {d.parent.name}": d for d in sorted((PATHS.output_root / "s3" / "runs").glob("*/predictions"))},
               **{f"S4 {d.parent.name}": d for d in sorted((PATHS.output_root / "s4" / "runs" / "seed0").glob("*/predictions"))}}

real_rows = []
for index, case in enumerate(development, 1):
    volumes = load_case(case, ("ct", "mask"))
    ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
    sources = {"bulk density": bulk_density_sct(mask)}
    for name, directory in PREDICTIONS.items():
        if (directory / f"{case.case_id}.mha").exists():
            sources[name] = read_volume(directory / f"{case.case_id}.mha").array
    for source, sct in sources.items():
        for row in case_identities(ct, mask, sct, spacing[0], hlut):
            real_rows.append({"case_id": case.case_id, "region": case.region, "source": source, **row})
    if index % 10 == 0:
        print(f"{index}/{len(development)} cases", flush=True)

real = pd.DataFrame(real_rows)
real.to_csv(OUTPUT_DIR / "real_identities.csv", index=False)
real_summary = {"task": TASK, "sources": sorted(real["source"].unique()), **summarize_identities(real)}
record("7", "Proposition 3 on patients: Parseval identity", f"max relative gap {real_summary['max_parseval_gap']:.1e}",
       real_summary["max_parseval_gap"] < 1e-8)
record("7", "Proposition 2 on patients: L²μ² + Lσ²l with the measured moments",
       f"max relative gap {real_summary['max_decomposition_gap']:.1e}", real_summary["max_decomposition_gap"] < 1e-8)
record("7", "Propositions 1, 1′ on patients: every ray within the sorted bounds",
       f"{real_summary['violations']} violations over {real_summary['rays']} rays", real_summary["violations"] == 0)

segments = real[real["check"] == "segments"]
display(segments.groupby(["source", "segment_mm", "beam"])[["mean_spr", "std_spr", "effective_length_mm", "rms_mm",
                                                            "white_rms_mm"]].median()
        .assign(coherence_gain=lambda f: f["rms_mm"] / f["white_rms_mm"]).round(4))
rays = real[real["check"] == "rays"]
display(rays.groupby(["source", "beam"])[["median_position", "max_position", "max_abs_mm", "worst_case_mm"]]
        .median().round(3))
print(f"{real_summary['cases']} development cases of task {TASK} · sources: {', '.join(real_summary['sources'])}")

## 8 · Verdicts

In [ ]:
verdicts = pd.DataFrame(checks, columns=["section", "claim", "evidence", "passed"])
report = {
    "seed": SEED,
    "hlut_fingerprint": hlut.fingerprint(),
    "real_data": real_summary,
    "checks": verdicts.to_dict(orient="records"),
    "all_passed": bool(verdicts["passed"].all()),
}
(OUTPUT_DIR / "s2_report.json").write_text(json.dumps(report, indent=2))
estimator_summary.to_csv(OUTPUT_DIR / "estimator_summary.csv")
display(verdicts)
print(f"{verdicts['passed'].sum()}/{len(verdicts)} checks passed")

## Consequences for S3–S5

Sections 1–6 hold on the synthetic benchmark, where the posterior is known; section 7 confirms the identities on
real patients and measures the coherence of real errors. The conclusions below are hypotheses that S3 and S4 test
on real data.

1. **Regression target.** The squared-error regressor in SPR space is the Bayes act for mean squared range error. It
   becomes arm R3 and the reference every other arm must beat. An L1 loss, the default in sCT work, carries a
   signed range bias wherever the posterior is asymmetric, with a sign set by the tissue pair.
2. **Role of the range loss.** At the Bayes act no loss can reduce mean squared range error further, so the projected
   loss can only help through finite capacity (reallocating residual errors away from coherent, low-frequency patterns)
   and through the tails. Its claimed benefit must be measured on CVaR at matched MAE, against R3 and the per-tissue bias
   correction R4.
3. **Tails.** Range-error tails depend on dependence between voxels that no point estimate encodes. They require a
   joint model (the stochastic-interpolant flow) or a direct VaR/ES head trained with the FZ score.
4. **Evaluation.** Generative sCT models must be compared with projected CRPS; voxel-wise CRPS is blind to exactly the
   property that governs range.
5. **Diagnostic for S3.** On real sCTs, the shuffling controls and the spectral split of the error decide which
   mechanism dominates: coherent tissue-level biases (range loss and bias correction matter) or interface blur
   (largely invisible to range).